# 0. Settings

In [ ]:
from IPython.display import clear_output

verbose = True # set to True for detailed output

# load huggingface secret if needed
#from google.colab import userdata
#userdata.get('HF_TOKEN')
HF_TOKEN = "YOUR_TOKEN_HERE"

In [ ]:
# set the base path, where the dataset is located, and results will be saved
base_path = "/user/pgrattacaso/" # edit if needed

# choose whether to download the dataset or not
download_dataset_flag = False

# choose the default label to apply to an ambiguous classification answer
ambiguous_class_default = True

# choose whether to save stats on a TXT file, to avoid relying solely on cell output
save_txt_flag = True

# choose whether to generate a final CSV report of per-sample results
generate_csv_flag = True


# -----


# select VideoLLaMa model
# available models: 2B, 7B
# on L40S with 44GB of RAM, both models work without CUDA OOM errors
model_name = "DAMO-NLP-SG/VideoLLaMA3-7B"

# select whether to load the original native-resolution videos
# or the downscaled videos with a max resolution of 448x448 pixels
use_downscaled_videos = False

# set the ideal FPS to load videos into the model
# videos with a lower natural framerate will be loaded at their native FPS value
# 5 fps is the threshold for recognizable human movement
# values >15 fsp are likely overkill
ideal_fps = 15

# set the maximum number of frames to be loaded at the same time
# if a video is longer than this, it will be split into multiple chunks
max_chunk_size = 200


# -----


# choose whether False Positives should affect the Mean Temporal Error
time_affects_FPs = False

# example:
# an originally False sample is predicted as True, with a 5 seconds start time
# - by setting time_affects_FPs = True, 5 seconds are added to the temporal error,
#   contributing to the MTE calculation
# - by setting time_affects_FPs = False, the temporal error is not affected


# choose whether False Negatives should affect the Mean Temporal Error
time_affects_FNs = False

# example:
# an originally True sample with a 3 seconds start time is predicted as False
# - by setting time_affects_FNs = True, 3 seconds are added to the temporal error,
#   contributing to the MTE calculation
# - by setting time_affects_FNs = False, the temporal error is not affected


# setting both flags to False ensures that the MTE is only computed on True Positives


# -----


# classification question
question1 = "Is a person seen littering or dumping garbage in a place where it's not allowed? Answer only with yes or no." # ORIGINAL
#question1 = "Is a person seen littering or dumping garbage in a place where it's not allowed? The video can be grayscale and can include visual limitations like blurring or distant actions. Answer only with yes or no." # DETAILED

# temporal localization question
question2 = "When does the littering or garbage dumping act start? Answer with the number of seconds from the start."


# 1. Data setup

In [ ]:
# download and extract dataset if needed (either mock or true one)

# use gdown to download
%pip install gdown
import os, gdown, zipfile


if download_dataset_flag:
    gdrive_file_ID = "YOUR_GOOGLE_DRIVE_FILE_ID" # dataset ID
    target_path = base_path + "DATASET" #'/content/DATASET'
    zip_filename = 'DATASET.zip'
    zip_filepath = base_path + zip_filename #f'/content/{zip_filename}'
    extracted_path = base_path + "DATASET" #'/content/DATASET'
    
    print(f"Attempting to download {gdrive_file_ID} from Google Drive...")
    #!gdown {gdrive_file_ID} -O {zip_filepath}
    gdown.download(f"https://drive.google.com/uc?id={gdrive_file_ID}", zip_filepath, quiet=False)
    
    # check if the zip file was downloaded successfully
    if os.path.exists(zip_filepath):
        print(f"Successfully downloaded {zip_filename}, extracting...")
        #!unzip -q {zip_filepath} -d /content/
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            zip_ref.extractall(extracted_path)
        if os.path.exists(extracted_path):
            print(f"Dataset extracted to {extracted_path}")
        else:
            print(f"Warning: expected directory '{extracted_path}' not found after extraction. Please check the zip file's internal structure.")
    else:
        print(f"Error: {zip_filename} not found after download attempt. Please check the Google Drive ID and permissions.")

else:
    print("Dataset download has been skipped, as selected by the user.\n")

In [ ]:
# function to process the dataset and create reference dictionaries

def process_dataset(dataset_path: str):
    """
    Creates reference dictionaries by processing the dataset.

    Args:
        dataset_path: The path of the root directory where the dataset is located.

    Returns:
        Three dictionaries.
        dict_labels: stores the True/False classification labels for each sample.
        dict_seconds: stores the seconds label for each positive sample.
        dict_truth_full: stores the seconds label for positive samples, and False for negative samples.
    """

    videos_dir = os.path.join(dataset_path, "videos")
    labels_dir = os.path.join(dataset_path, "labels")
    video_files = sorted([f for f in os.listdir(videos_dir) if f.endswith(".mp4")])
    label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith(".txt")])

    # check that all videos have a corresponding label
    video_indices = set([os.path.splitext(f)[0] for f in video_files])
    label_indices = set([os.path.splitext(f)[0] for f in label_files])

    if video_indices != label_indices:
        missing_labels = video_indices - label_indices
        missing_videos = label_indices - video_indices
        if missing_labels:
            print(f"Warning: The following videos are missing corresponding label files: {missing_labels}")
        if missing_videos:
            print(f"Warning: The following label files are missing corresponding videos: {missing_videos}")
        common_indices = video_indices.intersection(label_indices)
        print(f"Proceeding with {len(common_indices)} common video-label pairs.")
    else:
        print("All videos have corresponding label files and vice versa.")
        common_indices = video_indices

    # initialize dictionaries
    dict_labels = {}
    dict_seconds = {}
    dict_truth_full = {}

    # analyze labels
    for index in sorted(list(common_indices)):
        label_filepath = os.path.join(labels_dir, f"{index}.txt")
        video_filename = f"{index}.mp4"

        with open(label_filepath, 'r') as f:
            content = f.read().strip()

        if not content: # empty file, so negative label
            is_littering = False
            dict_truth_full[video_filename] = False

        else: # file not empty, so positive label
            is_littering = True
            try:
                # extract seconds, assuming it is an integer
                seconds = int(content)
            except ValueError:
                print(f"Error: Could not parse seconds from {label_filepath}. Setting seconds to default 0. Actual content: '{content}'")
                seconds = 0
            dict_seconds[video_filename] = seconds
            dict_truth_full[video_filename] = seconds

        dict_labels[video_filename] = is_littering

    return dict_labels, dict_seconds, dict_truth_full

In [ ]:
# process dataset

# directory where the dataset has been extracted into
dataset_path = base_path+"DATASET"
dict_labels, dict_seconds, dict_truth_full = process_dataset(dataset_path)

list_dataset_flag = False # output is very long

if verbose:
    if list_dataset_flag:
        print("\n--- dict_labels ---")
        for video, is_litter in dict_labels.items():
            print(f"{video}: {is_litter}")
        
        print("\n--- dict_seconds ---")
        for video, time in dict_seconds.items():
            print(f"{video}: {time}")
        
        print("\n--- dict_truth_full ---")
        for video, time in dict_truth_full.items():
            print(f"{video}: {time}")

___

### 1.1. Functions to parse LLM answers

In [ ]:
# function to parse classification answer

def parse_VLM_class(response: str, ambiguous_class_default: bool = False) -> bool:
    """
    Classifies the VLM response into True (littering) or False (no littering).

    Args:
        response: The answer provided by the VLM, as a string.
        ambiguous_class_default: The default class to apply to samples with ambiguous responses.

    Returns:
        A bool indicating the classification of the response.
    """

    response_lower = response.lower()

    # define keywords for positive and negative responses
    positive_keywords = ["yes", "littering", "litter", "garbage", "dumping", "present", "detected", "appears to be"]
    negative_keywords = ["no", "not", "does not", "doesn't", "do not", "don't", "any", "absent", "clear", "clean"]

    pos_num = 0
    neg_num = 0

    # check for positive keywords
    for keyword in positive_keywords:
        if keyword in response_lower:
            # positive, but check if it is part of a negative phrase
            if f"not {keyword}" in response_lower or f"no {keyword}" in response_lower:
                neg_num += 1
            else:
                pos_num += 1

    # check for negative keywords
    for keyword in negative_keywords:
        if keyword in response_lower:
            neg_num += 1

    # return result
    res = pos_num - neg_num
    if res != 0:
        return True if res > 0 else False

    # if there is no clear advantage
    print(f"Warning: Ambiguous LLM response received: '{response}'. Defaulting to {ambiguous_class_default}.")
    return ambiguous_class_default

In [ ]:
# function to parse seconds answer

import re

def parse_VLM_seconds(response: str) -> int:
    """
    Extracts the number of seconds (as an integer) from the VLM response,
    indicating when the littering action starts.

    Args:
        response: The answer provided by the VLM, as a string.

    Returns:
        An int indicating the classification of the response.
    """

    response_lower = response.lower()


    # 1. check for HH:MM:SS or MM:SS format
    
    # regex for HH:MM:SS or MM:SS (e.g., 01:23:45 or 23:45)
    hms_pattern = re.compile(r'(?:(\d{1,2}):)?(\d{1,2}):(\d{1,2})')
    hms_match = hms_pattern.search(response_lower)
    
    if hms_match:
        _, _, seconds_str = hms_match.groups()  # only consider seconds
        try:
            return int(seconds_str)
        except ValueError:
            pass  # continue with other methods if conversion fails

    #####
    # the behavior has been changed because LLM hallucinations
    # (e.g. answering 17:29:33 when the video was 20 seconds long)
    # caused the MTE statistic to grow uncontrollably
    # below is how the original method worked
    #####
    """
    # regex for HH:MM:SS or MM:SS (e.g., 01:23:45)
    hms_pattern = re.compile(r'(?:(\d{1,2}):)?(\d{1,2}):(\d{1,2})')
    hms_match = hms_pattern.search(response_lower)

    if hms_match:
        hours_str, minutes_str, seconds_str = hms_match.groups()
        try:
            total_seconds = 0
            if hours_str: # if hours were captured
                total_seconds += int(hours_str) * 3600
            total_seconds += int(minutes_str) * 60
            total_seconds += int(seconds_str)
            return total_seconds
        except ValueError:
            pass # continue with other methods if conversion fails
    """


    # 2. check for "X minutes and Y seconds"

    # regex to capture minutes and seconds explicitly stated
    # looks for:
    #  - one or more digits for minutes (\d+)\s*min(?:ute)?s?
    #  - optional "and"
    #  - one or more digits for seconds (\d+)\s*sec(?:ond)?s?
    min_sec_pattern = re.compile(r'(\d+)\s*min(?:ute)?s?\s*(?:and)?\s*(\d+)\s*sec(?:ond)?s?')
    min_sec_match = min_sec_pattern.search(response_lower)

    if min_sec_match:
        minutes_str, seconds_str = min_sec_match.groups()
        try:
            total_seconds = int(minutes_str) * 60 + int(seconds_str)
            return total_seconds
        except ValueError:
            pass # continue with other methods if conversion fails


    # 3. other cases

    # regex to find all numbers that could represent seconds
    # looks for:
    #  - one or more digits (\d+)
    #  - optionally followed by a decimal point and more digits (\\.?\\d*)
    # 'r' before the string denotes a raw string, which is good for regex patterns
    # the 're.IGNORECASE' flag makes the pattern case-insensitive
    pattern = re.compile(r'(\d+\.?\d*)\s*', re.IGNORECASE)

    # search for all matches in the response
    matches = pattern.findall(response_lower)

    # filter out unrecognizable matches
    potential_seconds = []
    for m in matches:
        try:
            num = float(m) # convert to float to handle decimals
            potential_seconds.append(num)
        except ValueError:
            continue # skip if conversion to float fails

    if potential_seconds:
        # prioritize numbers directly followed by 'sec'
        for match_obj in pattern.finditer(response_lower):
            num_str = match_obj.group(1)
            try:
                num_float = float(num_str)
                # check if 'sec' immediately follows the number
                if 'sec' in match_obj.group(0):
                    return int(num_float) # found a strong candidate, return it
            except ValueError:
                pass # continue to ambiguous answer if conversion fails

        # if no strong candidate (with 'second'/'sec') is found, default to the last number found
        return int(potential_seconds[-1]) #

    # if the number of seconds cannot be determined, fallback to 0
    print(f"Warning: Could not definitively extract start time from response: '{response}'.")
    return 0

# 2. Model setup

In [ ]:
!nvidia-smi

In [ ]:
# optimize cuda env settings before everything else
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# check python version, must be >=3.10
!python --version

In [ ]:
# if needed, uninstall existing versions to avoid conflicts
#!pip uninstall torch torchvision torchaudio -y

In [ ]:
# install torch and torchvision if needed
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# check versions
import torch, torchvision, torchaudio
print(torch.__version__)
print(torch.version.cuda)
print(torchvision.__version__)
print(torchaudio.__version__)

# tested working with versions:
# 2.5.1+cu121, 12.1, 0.20.1+cu121, 2.5.1+cu121

In [ ]:
# install requirements
!pip install transformers==4.46.3 accelerate==1.0.1
!pip install decord ffmpeg-python imageio opencv-python
!pip install ninja # useful for tricky flash-attn installation

In [ ]:
# in case you need to install flash-attn again, use this procedure

#!pip uninstall flash-attn -y
#!pip cache purge
#!git clone https://github.com/Dao-AILab/flash-attention.git
#%cd flash-attention
#!FLASH_ATTENTION_FORCE_BUILD=1 python setup.py install

In [ ]:
# clone the VideoLLaMa3 repo
!git clone https://github.com/DAMO-NLP-SG/VideoLLaMA3

# 3. Load model

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoProcessor


# optimization

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect() # optionally force garbage collection


# configuration

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if verbose:
    print(f"Using device: {device}")


# load model and processor

if verbose:
    print(f"Loading model: {model_name} onto {device}...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True, # allows the model to run custom code from Hugging Face
    device_map={"": device}, # maps the entire model to the specified device
    torch_dtype=torch.bfloat16, # use bfloat16 for reduced memory and faster inference if supported
    attn_implementation="flash_attention_2", # use flash_attention_2 for optimized attention
)

if verbose:
    print("Model loaded.")
    print("Loading processor...")
    
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)


if verbose:
    print("Processor loaded.")

### 3.1. Utility functions

In [ ]:
# function to compute various parameters about the video being loaded

import decord, math

def compute_video_parameters(video_path: str, ideal_fps: int, verbose: bool) -> tuple[int, int, int, int, float]:
    """
    Computes the parameters needed for the VLM to load a video.

    Args:
        video_path: Path to the video to be loaded.
        ideal_fps: Ideal FPS value to load the video, which acts as a cap.
        verbose: Flag to choose whether to display information messages or not.

    Returns:
        Two integers:
        - Native FPS of the video.
        - Target FPS to load the video.
        - Video duration in frames, at native FPS.
        - Video duration in frames, at target FPS.
        - Video duration in seconds.
        If all return values are 0, the video is considered not processable.
    """

    NOT_PROCESSABLE = (0, 0, 0, 0, 0)
    
    try:
        # 1. inspect video file with decord
        vr = decord.VideoReader(video_path)
        frames_native = len(vr)
        native_fps = vr.get_avg_fps()

        # 2. handle potential zero-division or empty videos
        if native_fps == 0 or frames_native == 0:
            print(f"WARNING: Cannot proccess video. '{video_path}' has {frames_native} frames and {native_fps} FPS. Skipping.")
            return NOT_PROCESSABLE

        duration_secs = frames_native / native_fps # video length in seconds
        
        # 3. signal that videos shorter than 0.5 seconds are not processable
        if duration_secs < 0.5:
            return NOT_PROCESSABLE
            
        # 4. compute target FPS and frames count at target FPS
        target_fps = ideal_fps if (native_fps >= ideal_fps) else math.floor(native_fps)
        frames_target = math.floor( frames_native * (target_fps/native_fps) )

        if verbose:
            print(f"Video duration: {duration_secs:.2f} s. Will be loaded at {target_fps} FPS, for a total of {frames_target} frames.")

    except Exception as e:
        print(f"ERROR: Could not compute parameters for video file '{video_path}'. It may be corrupted and not readable with decord. Skipping.\nError text: {e}")
        print(f"--- FINISHED PROCESSING VIDEO {video_path} ---\n\n")
        return native_fps, target_fps, frames_native, frames_target, duration_secs
    
    return native_fps, target_fps, frames_native, frames_target, duration_secs

In [ ]:
# helper function to format seconds into a readable string

def format_time(seconds):
    """Converts seconds into a readable string 'X h, Y min, Z sec'."""
    seconds = int(seconds) # working with whole numbers

    # if less than an hour
    if seconds < 3600:
        minutes = seconds // 60
        remaining_seconds = seconds % 60
        if minutes == 0:
             return f"{remaining_seconds} s"
        else:
             return f"{minutes} m, {remaining_seconds} s"
    
    # if more than an hour
    else:
        hours = seconds // 3600
        remainder_seconds = seconds % 3600
        minutes = remainder_seconds // 60
        remaining_seconds = remainder_seconds % 60
        return f"{hours} h, {minutes} m, {remaining_seconds} s"

# 4. Inference (classification)

In [ ]:
# start timer for full computation
import time
computation_beginning_time = time.time()

In [ ]:
dict_responses_q1 = dict() # to store the raw responses for the classification task
dict_predictions_full = dict() # to store final predictions to compare with ground truths

dict_videos_chunks = dict() # to store True chunks for split videos

if use_downscaled_videos:
    videos_dir = os.path.join(dataset_path, "videos_448")
else:
    videos_dir = os.path.join(dataset_path, "videos")
video_files = [f for f in os.listdir(videos_dir) if f.endswith(".mp4")]

print(f"Chosen settings for videos:\n- FPS: {ideal_fps}\n- max chunk size: {max_chunk_size} frames\n- use downscaled videos: {use_downscaled_videos}")

In [ ]:
# setup processing

total_videos = len(video_files)
videos_processed = 0
loop_start_time = time.time()


# loop through all the samples and perform inference

for video in video_files:
    
    if verbose:
        print(f"--- ({videos_processed+1}/{total_videos}) PROCESSING VIDEO {video} ---")

    video_path = videos_dir+"/"+video # template: "/content/dataset/videos/1.mp4"
    # check if video path is correct
    if not os.path.exists(video_path):
        print(f"ERROR: Video file not found at {video_path}. Skipping.")
        continue

    # compute required parameters
    native_fps, target_fps, frames_native, frames_target, duration_secs = compute_video_parameters(video_path, ideal_fps, verbose)

    
    # handle videos too short to be processable
    skip_processing = False
    if target_fps == 0 and frames_native == 0:
        # set default fallback response to False
        # MOTIVATION: video is too short to show an instance of human action
        skip_processing = True
        # simulate response
        response = "No." 
        if verbose:
                print("Video is too short to be processed correctly. Defaulting to False.")
                print(f"Sample: {video} - ground truth: {dict_truth_full[video]} - response: {response} ")
        dict_responses_q1[video] = response
    
    # handle eventual chunk splitting
    num_chunks = math.ceil(frames_target/max_chunk_size) # rounded up


    ##############################   1. CHUNK SPLITTING   ##############################
    if num_chunks > 1:

        if verbose:
            print(f"Video will be split into {num_chunks} chunks.")

        for chunk_idx in range(num_chunks):

            chunk_start_time = round( chunk_idx * max_chunk_size / target_fps, 2 ) # set chunk start time in seconds
            chunk_end_time = round( (chunk_idx+1) * max_chunk_size / target_fps, 2 ) # set chunk end time in seconds
            if verbose:
                print(f"- Processing chunk {chunk_idx+1} (from {chunk_start_time:.2f}s to {chunk_end_time:.2f}s)...")
            
            # define messages chain
            conversation = [
                {"role": "system", "content": "You are a helpful assistant."},
                {
                    "role": "user",
                    "content": [
                        {"type": "video", "video": {"video_path": video_path,
                                                    "fps": target_fps,
                                                    "start_time": chunk_start_time, # in seconds
                                                    "end_time": chunk_end_time, # in seconds
                                                    "max_frames": max_chunk_size}},
                        {"type": "text", "text": question1},
                    ]
                },
            ]
            
            # process inputs
            if not skip_processing:
                if verbose:
                    print("Processing inputs...")
                try:
                    inputs = processor(
                        conversation=conversation,
                        add_system_prompt=True,
                        add_generation_prompt=True,
                        return_tensors="pt"
                    )
                    inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
                    if "pixel_values" in inputs:
                        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
                    if verbose:
                        print("Inputs processed.")
            
                    # generate response
                    if verbose:
                        print("Generating response...")
                    output_ids = model.generate(**inputs, max_new_tokens=32) # response settings
                    response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
                    pred = parse_VLM_class(response)
    
                    # if chunk prediction is True, flag the video as True
                    if pred:
                        if verbose:
                            print(f"Chunk {chunk_idx+1} detected as True, flagging {video} as True.")
                            print(f"Sample: {video} - ground truth: {dict_truth_full[video]} - response: {response} ")
                        dict_responses_q1[video] = response # update responses dict
                        dict_videos_chunks[video] = chunk_idx # remember which chunk triggered the flag
                        break # video already flagged, no need to process other chunks
                    
                    # if chunk prediction is False, move on to the next chunk
                    # if last chunk, set video_flagged_true to False
                    else:
                        if chunk_idx == num_chunks-1:
                            if verbose:
                                print(f"All chunks detected as False, flagging {video} as False.")
                                print(f"Sample: {video} - ground truth: {dict_truth_full[video]} - response: {response} ")
                            dict_responses_q1[video] = response # update responses dict
                        
                except Exception as e:
                    print(f"\nAn error occurred: {e}")
                    if "CUDA out of memory" in str(e):
                        print("CUDA out of memory.")
                

    ##############################   2. SINGLE CHUNK   ##############################
    else:

        # define messages chain
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": {"video_path": video_path,
                                                "fps": target_fps,
                                                "max_frames": max_chunk_size}},
                    {"type": "text", "text": question1},
                ]
            },
        ]
        
        # process inputs
        if not skip_processing:
            if verbose:
                print("Processing inputs...")
            try:
                inputs = processor(
                    conversation=conversation,
                    add_system_prompt=True,
                    add_generation_prompt=True,
                    return_tensors="pt"
                )
                inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
                if "pixel_values" in inputs:
                    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
                if verbose:
                    print("Inputs processed.")
        
                # generate response
                if verbose:
                    print("Generating response...")
                output_ids = model.generate(**inputs, max_new_tokens=32) # response settings
                response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
                if verbose:
                    print(f"Sample: {video} - ground truth: {dict_truth_full[video]} - response: {response} ")
        
                # update responses dict
                dict_responses_q1[video] = response
        
            except Exception as e:
                print(f"\nAn error occurred: {e}")
                if "CUDA out of memory" in str(e):
                    print("CUDA out of memory.")


    # time estimation feedback
    videos_processed += 1
    elapsed_time = time.time() - loop_start_time
    avg_time_per_video = elapsed_time / videos_processed
    videos_remaining = total_videos - videos_processed
    if verbose:
        print(f"time elapsed: {format_time(elapsed_time)} | avg per video: {avg_time_per_video:.2f} s")
        if videos_remaining > 0:
            estimated_remaining_time = avg_time_per_video * videos_remaining
            print(f"estimated time remaining: {format_time(estimated_remaining_time)}")

    if verbose:
        print(f"--- FINISHED PROCESSING VIDEO {video} ---\n\n")

In [ ]:
# parse responses and save predictions

for video in video_files:
    pred = parse_VLM_class(dict_responses_q1[video])
    if verbose:
        print(f"{video} - GT: {dict_truth_full[video]} - PRED: {pred} - RESPONSE: {dict_responses_q1[video]}")
    dict_predictions_full[video] = pred

# 5. Inference (seconds)

In [ ]:
dict_responses_q2 = dict() # to store the raw responses for the classification task

print(f"Chosen settings for videos:\n- FPS: {ideal_fps}\n- max chunk size: {max_chunk_size} frames\n- use downscaled videos: {use_downscaled_videos}")

In [ ]:
# setup processing

total_videos = len(video_files)
videos_processed = 0
loop_start_time = time.time()


# loop through the samples and perform inference

for video in video_files:
    
    if verbose:
        print(f"--- ({videos_processed+1}/{total_videos}) PROCESSING VIDEO {video} ---")

    # skip videos predicted as False
    if dict_predictions_full[video] == False:
        videos_processed += 1
        if verbose:
            print("Video was predicted as False. Skipping temporal localization.")
            print(f"--- FINISHED PROCESSING VIDEO {video} ---\n\n")
        continue

    video_path = videos_dir+"/"+video # template: "/content/dataset/videos/1.mp4"
    # check if video path is correct
    if not os.path.exists(video_path):
        print(f"ERROR: Video file not found at {video_path}. Skipping.")
        continue
    
    # compute required parameters
    native_fps, target_fps, frames_native, frames_target, duration_secs = compute_video_parameters(video_path, ideal_fps, verbose)

    # video too short to be processable are defaulted to False
    # so none of those videos will undergo temporal localization inference
    
    # handle eventual chunk splitting
    num_chunks = math.ceil(frames_target/max_chunk_size) # rounded up


    ##############################   1. VIDEO WAS SPLIT IN CHUNKS  ##############################
    if num_chunks > 1:

        if verbose:
            flagged_chunk_idx = dict_videos_chunks[video]
            print(f"Video was split into {num_chunks} chunks, of which chunk {flagged_chunk_idx+1} was flagged as True.")

        chunk_start_time = round( flagged_chunk_idx * max_chunk_size / target_fps, 2 ) # set chunk start time in seconds
        chunk_end_time = round( (flagged_chunk_idx+1) * max_chunk_size / target_fps, 2 ) # set chunk end time in seconds
        if verbose:
            print(f"Processing chunk {flagged_chunk_idx+1} (from {chunk_start_time:.2f}s to {chunk_end_time:.2f}s)...")
        
        # define messages chain
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": {"video_path": video_path,
                                                "fps": target_fps,
                                                "start_time": chunk_start_time, # in seconds
                                                "end_time": chunk_end_time, # in seconds
                                                "max_frames": max_chunk_size}},
                    {"type": "text", "text": question1},
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": dict_responses_q1[video]}, # reconstruct conversation by using the model's original response
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question2},
                ]
            },
        ]

        # process inputs
        if verbose:
            print("Processing inputs...")
        try:
            inputs = processor(
                conversation=conversation,
                add_system_prompt=True,
                add_generation_prompt=True,
                return_tensors="pt"
            )
            inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
            if "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
            if verbose:
                print("Inputs processed.")

            # generate response
            if verbose:
                print("Generating response...")
            output_ids = model.generate(**inputs, max_new_tokens=32) # response settings
            response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
            if verbose:
                print(f"Sample: {video} - ground truth: {dict_truth_full[video]} - response: {response} ")

            # update responses dict
            dict_responses_q2[video] = response

        except Exception as e:
            print(f"\nAn error occurred: {e}")
            if "CUDA out of memory" in str(e):
                print("CUDA out of memory.")


    ##############################   2. VIDEO WAS NOT SPLIT   ##############################
    else:

        # define messages chain
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": {"video_path": video_path,
                                                "fps": target_fps,
                                                "max_frames": max_chunk_size}},
                    {"type": "text", "text": question1},
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": dict_responses_q1[video]}, # reconstruct conversation by using the model's original response
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question2},
                ]
            },
        ]

        # process inputs
        if verbose:
            print("Processing inputs...")
        try:
            inputs = processor(
                conversation=conversation,
                add_system_prompt=True,
                add_generation_prompt=True,
                return_tensors="pt"
            )
            inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
            if "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
            if verbose:
                print("Inputs processed.")

            # generate response
            if verbose:
                print("Generating response...")
            output_ids = model.generate(**inputs, max_new_tokens=32) # response settings
            response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
            if verbose:
                print(f"Sample: {video} - ground truth: {dict_truth_full[video]} - response: {response} ")

            # update responses dict
            dict_responses_q2[video] = response

        except Exception as e:
            print(f"\nAn error occurred: {e}")
            if "CUDA out of memory" in str(e):
                print("CUDA out of memory.")


    # time estimation feedback
    videos_processed += 1
    elapsed_time = time.time() - loop_start_time
    avg_time_per_video = elapsed_time / videos_processed
    videos_remaining = total_videos - videos_processed
    if verbose:
        print(f"time elapsed: {format_time(elapsed_time)} | avg per video: {avg_time_per_video:.2f} s")
        if videos_remaining > 0:
            estimated_remaining_time = avg_time_per_video * videos_remaining
            print(f"estimated time remaining: {format_time(estimated_remaining_time)}")

    if verbose:
        print(f"--- FINISHED PROCESSING VIDEO {video} ---\n\n")

In [ ]:
# parse responses and save predictions

for video in video_files:
    if dict_predictions_full[video] != False:
        
        pred = parse_VLM_seconds(dict_responses_q2[video]) # answer for single chunk

        # if video was split in chunks, add previous time
        if video in dict_videos_chunks:
            chunk_idx = dict_videos_chunks[video] # flagged chunk (idx starting from 0)
            previous_chunks_time = chunk_idx * math.floor(max_chunk_size/target_fps) # add seconds for past chunks
            pred = previous_chunks_time + pred # final chunk-aware prediction

        if verbose:
            print(f"{video} - GT: {dict_truth_full[video]} - PRED: {pred} - RESPONSE: {dict_responses_q2[video]}")
        dict_predictions_full[video] = pred

# 6. Get results

In [ ]:
# function to compare the ground truth dict and the predictions dict

def compare_dictionaries(dict_truth_full: dict,
                         dict_predictions_full: dict,
                         verbose: bool = False
                        ) -> tuple[dict, float, float, float, float]:
    """
    Compares two dictionaries (ground truth and inference) to calculate
    precision, recall, and mean temporal error (MTE).

    Args:
        dict_truth_full:    The ground truth dictionary. Keys are video filenames (str),
                            values are either a float/int (seconds) for positive samples
                            or False for negative samples.
        dict_predictions_full:  The inference dictionary. Keys are video filenames (str),
                                values are either a float/int (seconds) for positive predictions
                                or False for negative predictions.

    Returns:
        - A tuple containing (stats, precision, recall, fscore, MTE).
        "stats" is a dictionary containing the following keys: TP, FP, TN, FN.
        Returns ({}, 0.0, 0.0, 0.0, 0.0) if no common samples or no positive ground truth.
    """

    # check if both dictionaries have the same keys
    if set(dict_truth_full.keys()) != set(dict_predictions_full.keys()):
        print("Error: Dictionaries have different keys, cannot generate CSV report.")
        return dict(), 0.0, 0.0, 0.0, 0.0


    TPs = 0; FPs = 0; TNs = 0; FNs = 0
    temporal_errors = [] # list to store absolute differences for MTE

    # assuming both dictionaries have the same keys for comparison
    # if not, please handle mismatches
    # we'll only compare common keys to ensure a fair evaluation
    common_keys = set(dict_truth_full.keys()).intersection(set(dict_predictions_full.keys()))
    if not common_keys:
        print("Warning: No common video samples found between ground truth and inference dictionaries.")
        return dict(), 0.0, 0.0, 0.0, 0.0

    if verbose:
        print(f"Comparing {len(common_keys)} common video samples...")

    common_keys = list(common_keys) # convert to list
    common_keys.sort() # sort in ascending order
    
    for video_key in common_keys:
        ground_truth_val = dict_truth_full[video_key]
        inference_val = dict_predictions_full[video_key]

        # determine if ground truth and inference value are truly positive or negative
        is_ground_truth_pos = (ground_truth_val is not False)
        is_inference_pos = (inference_val is not False)

        if is_ground_truth_pos and is_inference_pos:
            TPs += 1
            # ensure both values are numerical for temporal error calculation
            if isinstance(ground_truth_val, (int, float)) and isinstance(inference_val, (int, float)):
                temporal_errors.append(abs(ground_truth_val - inference_val))
            else:
                print(f"Warning: Non-numeric values for temporal error in {video_key} (GT: {ground_truth_val}, Inf: {inference_val}). Skipping temporal error for this sample.")
        elif not is_ground_truth_pos and is_inference_pos:
            FPs += 1
            # if selected, let FPs account for temporal error
            if time_affects_FPs:
                # ensure inference value is numerical for temporal error calculation
                if isinstance(inference_val, (int, float)):
                    temporal_errors.append(inference_val)
                else:
                    print(f"Warning: Non-numeric inference value for temporal error in {video_key} (GT: {ground_truth_val}, Inf: {inference_val}). Skipping temporal error for this sample.")
        elif is_ground_truth_pos and not is_inference_pos:
            FNs += 1
            # if selected, let FNs account for temporal error
            if time_affects_FNs:
                # ensure ground truth value is numerical for temporal error calculation
                if isinstance(ground_truth_val, (int, float)):
                    temporal_errors.append(ground_truth_val)
                else:
                    print(f"Warning: Non-numeric ground truth value for temporal error in {video_key} (GT: {ground_truth_val}, Inf: {inference_val}). Skipping temporal error for this sample.")
        else:
            TNs += 1
            # TNs don't contribute to precision/recall directly nor to MTE,
            # so there's no true need to count TNs for these metrics

    # calculate precision
    # precision = TP / (TP + FP)
    precision = 0.0
    if (TPs + FPs) > 0:
        precision = TPs / (TPs + FPs)
    else:
        print("Note: Denominator for Precision is zero (no positive predictions). Precision set to 0.")


    # calculate recall
    # recall = TP / (TP + FN)
    recall = 0.0
    if (TPs + FNs) > 0:
        recall = TPs / (TPs + FNs)
    else:
        print("Note: Denominator for Recall is zero (no actual positive samples). Recall set to 0.")


    # calculate F-score
    # fscore = 2 * precision * recall / (precision + recall)
    fscore = 0.0
    if (precision + recall) > 0:
        fscore = 2 * precision * recall / (precision + recall)
    else:
        print("Note: Denominator for F-score is zero (Precision and Recall are both 0). F-score set to 0.")


    # calculate mean temporal error (MTE)
    # MTE is the average of absolute temporal errors for TPs
    mean_temporal_error = 0.0
    if temporal_errors: # check if there are any TPs with measurable errors
        mean_temporal_error = sum(temporal_errors) / len(temporal_errors)
    else:
        print("Note: No True Positives with measurable temporal error found. MTE set to 0.0.")

    if verbose:
        print("\n--- Comparison Results ---")
        print(f"True Positives (TP): {TPs}")
        print(f"False Positives (FP): {FPs}")
        print(f"True Negatives (TN): {TNs}")
        print(f"False Negatives (FN): {FNs}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F-score: {fscore:.4f}")
        print(f"Mean Temporal Error (MTE): {mean_temporal_error:.4f} seconds")

    # construct return dictionary
    stats = dict()
    stats["TP"] = TPs; stats["FP"] = FPs; stats["TN"] = TNs; stats["FN"] = FNs
    
    return stats, precision, recall, fscore, mean_temporal_error

In [ ]:
# function to generate the final CSV report of per-sample results

import csv

def generate_csv_report(dict_truth_full: dict,
                        dict_predictions_full: dict,
                        filename: str = "CSV_report.csv"
                       ):
    """
    Compiles and exports a CSV file storing all the information
    relative to each sample of the dataset, including:
    0) video filename
    1) ground truth classification label
    2) ground truth temporal label
    3) inference respose to classification question
    4) prediction for classification
    5) inference respose to temporal localization question
    6) prediction for temporal localization
    7) classification prediction correct True/False
    8) temporal localization prediction correct True/False
    
    Args:
        dict_truth_full:    The ground truth dictionary. Keys are video filenames (str),
                            values are either a float/int (seconds) for positive samples
                            or False for negative samples.
        dict_predictions_full:  The inference dictionary. Keys are video filenames (str),
                                values are either a float/int (seconds) for positive predictions
                                or False for negative predictions.
        filename:   The name of the CSV file to be created (str).
    """

    # check if both dictionaries have the same keys
    if set(dict_truth_full.keys()) != set(dict_predictions_full.keys()):
        print("Error: Dictionaries have different keys, cannot generate CSV report.")
        return False
    
    # create a CSV file with the required columns
    with open(filename, 'w', newline='') as csvfile:
        fieldnames = ['Video', 'Label_CL', 'Label_TL', 'Response_CL', 'Prediction_CL', 'Response_TL', 'Prediction_TL', 'Correct_CL', 'Correct_TL']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        # iterate over the video keys and populate the CSV
        for video_key in dict_truth_full:
            prediction_CL = False if dict_predictions_full[video_key] is False else True
            response_TL = "" if dict_predictions_full[video_key] is False else dict_responses_q2[video_key]
            prediction_TL = False if dict_predictions_full[video_key] is False else dict_predictions_full[video_key]

            writer.writerow({
                'Video': video_key,
                'Label_CL': dict_labels[video_key],
                'Label_TL': dict_truth_full[video_key],
                'Response_CL': dict_responses_q1[video_key],
                'Prediction_CL': prediction_CL,
                'Response_TL': response_TL,
                'Prediction_TL': prediction_TL,
                'Correct_CL': dict_labels[video_key] == prediction_CL,
                'Correct_TL': dict_truth_full[video_key] == dict_predictions_full[video_key]
            })

    return

In [ ]:
# compare dicts and get results
stats, precision, recall, fscore, mte = compare_dictionaries(dict_truth_full, dict_predictions_full, verbose=verbose)
print(f"\nFinal calculated metrics: Precision={precision:.4f}, Recall={recall:.4f}, F-score={fscore:.4f}, MTE={mte:.4f}")


# shorten model name, by getting the text after '/'
short_model_name = model_name.split('/')[1].replace("-", "_")


# if requested, save stats to a TXT file
if save_txt_flag:
    base_filename = base_path + f"stats_{short_model_name}_{ideal_fps}fps_{max_chunk_size}chunk"
    filename = base_path + f"stats_{short_model_name}_{ideal_fps}fps_{max_chunk_size}chunk.txt"
    # find an available name without overwriting anything
    if os.path.exists(filename):
        i = 1
        while True:
            filename = base_filename + str(i) + ".txt"
            if not os.path.exists(filename):
                break
            i += 1
    # save TXT
    with open(filename, "w") as f:
        f.write(f"ideal_fps = {ideal_fps}\nmax_chunk_size = {max_chunk_size}\nuse_downscaled_videos = {use_downscaled_videos}\n")
        f.write("TP: " + str(stats["TP"]) + "\n")
        f.write("FP: " + str(stats["FP"]) + "\n")
        f.write("TN: " + str(stats["TN"]) + "\n")
        f.write("FN: " + str(stats["FN"]) + "\n\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall: {recall:.4f}\n")
        f.write(f"F-score: {fscore:.4f}\n")
        f.write(f"MTE: {mte:.4f} seconds\n")
    print("Stats saved to " + filename)


# if requested, generate CSV file with detailed results
if generate_csv_flag:
    base_filename = base_path + f"report_{short_model_name}_{ideal_fps}fps_{max_chunk_size}chunk"
    filename = base_path + f"report_{short_model_name}_{ideal_fps}fps_{max_chunk_size}chunk.csv"
    # find an available name without overwriting anything
    if os.path.exists(filename):
        i = 1
        while True:
            filename = base_filename + str(i) + ".csv"
            if not os.path.exists(filename):
                break
            i += 1
    generate_csv_report(dict_truth_full, dict_predictions_full, filename)

# check CSV creation
if generate_csv_flag and os.path.exists(filename):
    print(f"CSV report saved to {filename}")


# show total computation time
final_time = time.time() - computation_beginning_time
print(f"Total time: {format_time(final_time)}\n")